In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/harshshinde8/movies-csv/movies.csv


In [1]:
# Importing important libraries required for the recommendation/similarity system

import numpy as np
# Used for numerical computations and array operations

import pandas as pd
# Used for data loading, manipulation, and analysis

import seaborn as sns
# Used for statistical data visualization

import matplotlib.pyplot as plt
# Used for plotting graphs and charts

import difflib
# Used for finding close matches between strings
# Helpful when user input contains spelling mistakes
# Example:
# "Avngers" -> "Avengers"

from sklearn.feature_extraction.text import TfidfVectorizer
# Converts textual data into TF-IDF numerical feature vectors
# Helps machine learning algorithms understand text data

from sklearn.metrics.pairwise import cosine_similarity
# Used to calculate similarity between vectors/documents
# Returns similarity scores between 0 and 1
# Commonly used in recommendation systems and NLP projects

print("All the necessary libraries are imported !!!")

All the necessary libraries are imported !!!


In [3]:
# Loading the movies dataset from the CSV file using pandas

dataset = pd.read_csv("/kaggle/input/datasets/harshshinde8/movies-csv/movies.csv")

# Displays the first 5 rows of the dataset
# Helps us understand:
# - available columns
# - movie information
# - dataset structure
# - sample records

dataset.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


 budget -> total money spent to produce the movie

 genres -> movie categories like Action, Comedy, Thriller

 homepage -> official website of the movie

 id -> unique identifier for each movie

 keywords -> important tags/words related to the movie

 original_language -> original language of the movie

 original_title -> original name/title of the movie

 overview -> short summary/story of the movie

 popularity -> popularity score of the movie

 runtime -> total movie duration in minutes

 spoken_languages -> languages spoken in the movie

 status -> release status of the movie

 tagline -> short promotional slogan of the movie

 title -> main/display title of the movie

 vote_average -> average user rating of the movie

 vote_count -> total number of ratings/votes

 cast -> actors and actresses in the movie

 crew -> people involved in movie production

 director -> director of the movie

In [6]:
# Getting the total number of rows and columns in the dataset

row, column = dataset.shape

# Displaying the number of rows
# Rows represent total movie records in the dataset

print("The number of rows : ", row)

# Displaying the number of columns
# Columns represent different movie features/attributes

print("The number of columns : ", column)

The number of rows :  4803
The number of columns :  24


In [11]:
# Selecting important/relevant text features for the recommendation system
# These columns contain useful textual information about movies
# which helps in finding movie similarities

features = ['genres', 'tagline', 'director', 'cast', 'keywords']

# Displaying the selected feature names

print("The selected features are:\n", features)

The selected features are:
 ['genres', 'tagline', 'director', 'cast', 'keywords']


In [12]:
# Handling missing/null values in the selected feature columns

for feature in features:
    
    # Replacing missing (NaN) values with empty strings ''
    # This prevents errors during text processing and TF-IDF vectorization
    
    dataset[feature] = dataset[feature].fillna('')

# Confirmation message after handling missing values

print("Successfully handled missing values")

Successfully handled missing values


In [14]:
# Checking the number of missing/null values in each column of the dataset

# isnull() -> identifies missing values
# sum() -> counts total missing values column-wise

dataset.isnull().sum()

index                      0
budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                    0
title                      0
vote_average               0
vote_count                 0
cast                       0
crew                       0
director                   0
dtype: int64

In [15]:
# Combining selected text features into one single text column

combined_features = (
    
    dataset['genres'] + ' ' +
    
    dataset['tagline'] + ' ' +
    
    dataset['director'] + ' ' +
    
    dataset['cast'] + ' ' +
    
    dataset['keywords']
)

# Displaying first 5 combined rows

combined_features.head()

0    Action Adventure Fantasy Science Fiction Enter...
1    Adventure Fantasy Action At the end of the wor...
2    Action Adventure Crime A Plan No One Escapes S...
3    Action Crime Drama Thriller The Legend Ends Ch...
4    Action Adventure Science Fiction Lost in our w...
dtype: object

In [19]:
# Converting the combined textual movie information into numerical feature vectors

# Creating an object of TfidfVectorizer
# TF-IDF converts text data into numerical values based on word importance

vectorizer = TfidfVectorizer()

# fit_transform() performs two operations:
#
# 1. fit():
#    Learns important vocabulary and IDF values from the text data
#
# 2. transform():
#    Converts the text data into numerical TF-IDF vectors
#
# The output will be a sparse matrix containing numerical feature vectors
# for all movies

features_vectors = vectorizer.fit_transform(combined_features)

print("The combined features are transformed to feature vectors")
print(features_vectors)

The combined features are transformed to feature vectors
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 124266 stored elements and shape (4803, 17318)>
  Coords	Values
  (0, 201)	0.07860022416510505
  (0, 274)	0.09021200873707368
  (0, 5274)	0.11108562744414445
  (0, 13599)	0.1036413987316636
  (0, 5437)	0.1036413987316636
  (0, 4945)	0.24025852494110758
  (0, 15261)	0.07095833561276566
  (0, 16998)	0.1282126322850579
  (0, 11192)	0.09049319826481456
  (0, 11503)	0.27211310056983656
  (0, 7755)	0.1128035714854756
  (0, 2432)	0.17272411194153
  (0, 13349)	0.15021264094167086
  (0, 17007)	0.23643326319898797
  (0, 17290)	0.20197912553916567
  (0, 13319)	0.2177470539412484
  (0, 14064)	0.20596090415084142
  (0, 16668)	0.19843263965100372
  (0, 14608)	0.15150672398763912
  (0, 8756)	0.22709015857011816
  (0, 10229)	0.16058685400095302
  (0, 13024)	0.1942362060108871
  (0, 3678)	0.21392179219912877
  (0, 3065)	0.22208377802661425
  (0, 5836)	0.1646750903586285
  :	:
  (4801, 

In [22]:
# Calculating similarity scores between all movies using cosine similarity

# Cosine similarity measures how similar two movie vectors are
# by calculating the cosine of the angle between them
#
# Similarity score range:
# 1  -> very similar movies
# 0  -> completely different movies
#
# Higher cosine similarity means movies share similar:
# - genres
# - cast
# - director
# - keywords
# - storyline-related information

similarity = cosine_similarity(features_vectors)

# Displaying the similarity matrix
# Each row and column represent movies
# The values indicate similarity scores between movies

print(similarity)

# Displaying the shape of the similarity matrix
# Example:
# (4803, 4803)
#
# Means:
# 4803 movies compared with all 4803 movies

print(similarity.shape)

# Confirmation message

print("Successfully got cosine similarity scores!!")

[[1.         0.07219487 0.037733   ... 0.         0.         0.        ]
 [0.07219487 1.         0.03281499 ... 0.03575545 0.         0.        ]
 [0.037733   0.03281499 1.         ... 0.         0.05389661 0.        ]
 ...
 [0.         0.03575545 0.         ... 1.         0.         0.02651502]
 [0.         0.         0.05389661 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02651502 0.         1.        ]]
(4803, 4803)
Successfully got cosine similarity scores!!


In [29]:
# Taking movie name input from the user

# The user enters their favourite movie name
# The recommendation system will use this movie
# to find similar/recommended movies

movie_name_input = input('Enter your favourite movie name : ')

Enter your favourite movie name :  Pirates of the Caribbean


In [26]:
# Creating a list containing all movie titles from the dataset

# ['title'] selects the movie title column
# .tolist() converts the pandas column into a normal Python list

list_of_titles = dataset['title'].tolist()

# This list will be used to:
# - search movie names
# - find close matches
# - recommend similar movies

In [33]:
 # Finding the closest matching movie name from the dataset

# difflib.get_close_matches() compares the user input
# with all movie titles and returns the most similar matches
#
# Helpful when:
# - user makes spelling mistakes
# - movie name is partially typed
#
# Example:
# "Avngers" -> "Avengers"

find_the_closest_match = difflib.get_close_matches(
    movie_name_input,
    list_of_titles
)

# Displaying the closest matched movie names

print(find_the_closest_match)

# Selecting the best/first closest match from the list
# [0] retrieves the first movie name from matched results

close_match = find_the_closest_match[0]



["Pirates of the Caribbean: At World's End", "Pirates of the Caribbean: Dead Man's Chest", 'Pirates of the Caribbean: On Stranger Tides']
